In [1]:
import numpy as np


def read_latency_data(file, ignore):
    data = {"end_to_end": [], "service": [], "slo": []}
    with open(file, "r") as file:
        for line in file.readlines():
            span, slo, latency = line.strip().split(",")
            assert span in data.keys()
            data[span].append(int(latency))
            if span == "end_to_end":
                data["slo"].append(int(slo))
    for key in data.keys():
        n_ignore = int(len(data[key]) * ignore)
        data[key] = data[key][n_ignore:]
    return data


def get_pctl(data, pctl):
    data = np.sort(data)
    tail = np.percentile(data, pctl).astype(int)
    return tail


def get_slo_vio(data, slo):
    cnt = 0
    for i in range(len(data)):
        if data[i] > slo[i]:
            cnt += 1
    return cnt / len(data)

In [5]:
for file in ["fcfs.csv", "masa.csv"]:
    print(file)

    data = read_latency_data(file, ignore=0.01)
    for pctl in [50, 90, 99, 99.9]:
        tail = get_pctl(data["end_to_end"], pctl)
        print(f"{pctl}: {tail}")

    slo_vio = get_slo_vio(data["end_to_end"], data["slo"])
    print(f"SLO violation: {slo_vio*100:.2f}%")

    # for pctl in [50, 90, 99, 99.9]:
    #     tail = get_pctl(data["service"], pctl)
    #     print(f"{pctl}: {tail}")

    print()

fcfs.csv
50: 98030
90: 144477
99: 192309
99.9: 231278
SLO violation: 48.62%

masa.csv
50: 68541
90: 190680
99: 251645
99.9: 316805
SLO violation: 17.96%

